# Maximum detectable redshift of SN

In [12]:
import sncosmo
import numpy as np
from astropy.cosmology import Planck18 as cosmo
from rich import print

def estimate_max_redshift(sn_type, limiting_mag, band, mpeak):
    """
    Estimate the maximum redshift of a detectable supernova (SN).
    
    Parameters:
        sn_type (str): The type of supernova (e.g., 'Ia', 'Ibc', 'II').
        limiting_mag (float): The limiting magnitude of the observation.
        band (str): The photometric band (e.g., 'bessellb', 'bessellv', 'bessellr', 'besselli').
    
    Returns:
        max_redshift (float): The maximum redshift at which the SN is detectable.
        max_phase (float): The phase (in days) at which the SN reaches maximum brightness.
    """
    # Load the SN model based on the SN type
    if sn_type.lower() == 'ia':
        model = sncosmo.Model(source='salt2')
    elif sn_type.lower() == 'ibc':
        model = sncosmo.Model(source='nugent-sn1bc')
    elif sn_type.lower() == 'ii':
        model = sncosmo.Model(source='nugent-sn2p')
    else:
        raise ValueError("Unsupported SN type. Supported types: 'Ia', 'Ibc', 'II'.")

    # Set the model parameters
    model.set(z=0.01)  # Start with a small redshift
    model.set_source_peakabsmag(mpeak, band, 'ab')  # Typical absolute magnitude for SN Ia

    # Find the phase of maximum brightness
    time = np.linspace(-20, 50, 100)  # Time range in days
    flux = model.bandflux(band, time)
    max_phase = time[np.argmax(flux)]

    # Estimate the maximum redshift
    max_redshift = 0.01
    while True:
        model.set(z=max_redshift,t0=0.,amplitude=1)
        model.set_source_peakabsmag(mpeak, band, 'ab') 
        apparent_mag = model.bandmag(band, 'ab', max_phase)
        if apparent_mag > limiting_mag:
            break
        max_redshift += 0.01

    return max_redshift - 0.01, max_phase

# Example usage
sn_type = 'Ibc'
limiting_mag = 21.5
band = 'sdssr'
mpeak = -21

max_redshift, max_phase = estimate_max_redshift(sn_type, limiting_mag, band, mpeak=mpeak)
print(f"Model SN-{sn_type} (peak abs mag = {mpeak}) with limiting magnitude {limiting_mag} in band {band}:")
print(f"Maximum redshift: {max_redshift:.2f}")

Model SN-Ibc (peak abs mag = -21) with limiting magnitude 21.5 in band sdssr:

Maximum redshift: 0.52